# BOLLBOT – retrain the YOLOv8 cotton boll detector

Run the cells **from top to bottom**, one at a time (click a cell, then press **Shift + Enter**).

**Before you start:** click **Runtime → Change runtime type → T4 GPU → Save**.

At the end, the notebook downloads **`bollbot_results.zip`**. Send that ZIP file to Claude. It contains every number and figure the paper needs.

Dataset: *Cotton Boll Detection* by Suwathi M, Roboflow Universe, licence CC BY 4.0
(https://universe.roboflow.com/suwathi-m/cotton-boll-detection).

## Step 1 – Check the GPU and install the libraries
You should see **Tesla T4** printed. If it says no GPU, redo the runtime step above.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

## Step 2 – Save results to Google Drive (protects your work if Colab disconnects)
A pop-up will ask you to allow access to Google Drive. Click **Allow**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/bollbot_runs'
import os; os.makedirs(RUN_DIR, exist_ok=True)
print('Results will be saved in:', RUN_DIR)

## Step 3 – Download the dataset
1. Make a free account at https://roboflow.com (Sign in with Google is fine).
2. Open https://app.roboflow.com → click your name (top left) → **Settings** → **API Keys** → copy the **Private API Key**.
3. Paste it between the quotes below, then run the cell.

In [ ]:
API_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY_HERE"
VERSION = 1   # dataset version on Roboflow; leave as 1 unless told otherwise

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("suwathi-m").project("cotton-boll-detection")
dataset = project.version(VERSION).download("yolov8", location="/content/cotton_dataset")
DATA_YAML = dataset.location + "/data.yaml"

# make the dataset paths absolute so YOLOv8 always finds the images
import yaml, os
cfg = yaml.safe_load(open(DATA_YAML))
for split, folder in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
    if os.path.isdir(f"{dataset.location}/{folder}/images"):
        cfg[split] = f"{dataset.location}/{folder}/images"
    else:
        cfg.pop(split, None)
yaml.safe_dump(cfg, open(DATA_YAML, 'w'))
print("Dataset ready:", DATA_YAML)
print(open(DATA_YAML).read())

## Step 4 – Count the images and labels (this fills Table 2 of the paper)

In [ ]:
import os, glob, yaml, collections, csv
cfg = yaml.safe_load(open(DATA_YAML))
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
root = os.path.dirname(DATA_YAML)
rows = []
for split in ['train', 'valid', 'test']:
    imgs = glob.glob(f'{root}/{split}/images/*')
    counts = collections.Counter()
    for lf in glob.glob(f'{root}/{split}/labels/*.txt'):
        for line in open(lf):
            if line.strip():
                counts[int(line.split()[0])] += 1
    rows.append([split, len(imgs)] + [counts[i] for i in range(len(names))])
header = ['split', 'images'] + names
with open(f'{RUN_DIR}/dataset_stats.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(header); w.writerows(rows)
print(' | '.join(header))
for r in rows: print(' | '.join(map(str, r)))

## Step 5 – Train the model (the long step: about 1–2 hours)
Keep this browser tab open. If Colab disconnects, the finished epochs are already saved in Google Drive; tell Claude and you'll be told how to resume.

In [ ]:
EPOCHS = 30     # 30 epochs fits in a free Colab session
BATCH = 16
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=BATCH, seed=0,
            project=RUN_DIR, name='bollbot_yolov8s', exist_ok=True, plots=True)

## Step 6 – Measure accuracy on the validation and test sets (this fills Table 4 and the results text)

In [ ]:
import json
best = f'{RUN_DIR}/bollbot_yolov8s/weights/best.pt'
model = YOLO(best)
report = {}
for split in ['val', 'test']:
    if split not in yaml.safe_load(open(DATA_YAML)):
        print('No', split, 'set in this dataset version - skipped'); continue
    m = model.val(data=DATA_YAML, split=split, imgsz=640, conf=0.001, iou=0.6,
                  project=RUN_DIR, name=f'eval_{split}', exist_ok=True, plots=True)
    b = m.box
    P, R = float(b.mp), float(b.mr)
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    per_class = {}
    for j, ci in enumerate(b.ap_class_index):
        per_class[m.names[int(ci)]] = {'precision': float(b.p[j]), 'recall': float(b.r[j]),
                                       'AP50': float(b.ap50[j]), 'AP50_95': float(b.ap[j])}
    report[split] = {'precision': P, 'recall': R, 'F1': F1, 'mAP50': float(b.map50),
                     'mAP50_95': float(b.map), 'per_class': per_class, 'speed_ms_per_image': m.speed}
    print(f'\n=== {split.upper()} ===  P={P:.3f}  R={R:.3f}  F1={F1:.3f}  mAP50={b.map50:.3f}  mAP50-95={b.map:.3f}')
    for k, v in per_class.items():
        print(f'  {k:40s} AP50={v["AP50"]:.3f}  P={v["precision"]:.3f}  R={v["recall"]:.3f}')
    print('  speed (ms per image on Colab GPU):', m.speed)
json.dump(report, open(f'{RUN_DIR}/metrics_report.json', 'w'), indent=2)

## Step 7 – Make example detection pictures (for Fig. 2 of the paper)

In [ ]:
test_imgs = sorted(glob.glob(f'{root}/test/images/*'))[:8]
model.predict(test_imgs, conf=0.5, imgsz=640, save=True,
              project=RUN_DIR, name='examples', exist_ok=True)
print('Saved example pictures in', f'{RUN_DIR}/examples')

## Step 8 – Download everything as one ZIP and send it to Claude

In [ ]:
import shutil
from google.colab import files
zip_base = '/content/bollbot_results'
tmp = '/content/bollbot_results_dir'
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(RUN_DIR, tmp, ignore=shutil.ignore_patterns('*.pt'))   # leave out the large model weights
shutil.copy(DATA_YAML, tmp)
shutil.make_archive(zip_base, 'zip', tmp)
files.download(zip_base + '.zip')